In [ ]:
import collections
import random
import re
import torch
from torch.utils import data

from d2l import torch as d2l 

# Giả sử đã download file timemachine.txt
with open('./.data/timemachine.txt') as f:
    raw_text = f.read()

print(raw_text[:60])


The Time Machine, by H. G. Wells [1898]




I


The Time Tra


In [8]:
class TimeMachine(d2l.DataModule): #@save
    """The Time Machine dataset."""
    def _download(self):
        fname = d2l.download(d2l.DATA_URL + 'timemachine.txt', self.root,
                             '090b5e7e70c295757f55df93cb0a180b9691891a')
        with open(fname) as f:
            return f.read()

data = TimeMachine()
raw_text = data._download()
raw_text[:60]


'The Time Machine, by H. G. Wells [1898]\n\n\n\n\nI\n\n\nThe Time Tra'

In [10]:
def preprocess(text):
    """Loại bỏ tất cả ký tự không phải chữ cái, chuyển về lowercase."""
    return re.sub('[^A-Za-z]+', ' ', text).lower()

text = preprocess(raw_text)
print(text[:60])


the time machine by h g wells i the time traveller for so it


In [12]:
def tokenize_words(text):
    return text.split()

words = tokenize_words(text)
print(words[:10])


['the', 'time', 'machine', 'by', 'h', 'g', 'wells', 'i', 'the', 'time']


In [13]:
def tokenize_chars(text):
    return list(text)

chars = tokenize_chars(text)
print(','.join(chars[:30]))


t,h,e, ,t,i,m,e, ,m,a,c,h,i,n,e, ,b,y, ,h, ,g, ,w,e,l,l,s, 


In [14]:
class Vocab:
    """Vocabulary for text."""
    def __init__(self, tokens=[], min_freq=0, reserved_tokens=[]):
        # Nếu tokens là list 2D (list of lines), flatten thành 1D
        if tokens and isinstance(tokens[0], list):
            tokens = [token for line in tokens for token in line]
        
        # Đếm tần suất từng token
        counter = collections.Counter(tokens)
        # Sắp xếp giảm dần theo frequency
        self.token_freqs = sorted(counter.items(), 
                                   key=lambda x: x[1], reverse=True)
        
        # Xây dựng idx_to_token list
        # Bắt đầu bằng <unk> + reserved tokens + tokens thỏa min_freq
        self.idx_to_token = list(sorted(set(
            ['<unk>'] + reserved_tokens + 
            [token for token, freq in self.token_freqs 
             if freq >= min_freq]
        )))
        # Xây dựng token_to_idx dict (reverse mapping)
        self.token_to_idx = {token: idx 
                              for idx, token in enumerate(self.idx_to_token)}
    
    def __len__(self):
        return len(self.idx_to_token)
    
    def __getitem__(self, tokens):
        """Token(s) → Index(es). Từ không biết → index của <unk>."""
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]
    
    def to_tokens(self, indices):
        """Index(es) → Token(s)."""
        if hasattr(indices, '__len__') and len(indices) > 1:
            return [self.idx_to_token[int(index)] for index in indices]
        return self.idx_to_token[indices]
    
    @property
    def unk(self):
        """Index cho token không xác định."""
        return self.token_to_idx['<unk>']


In [16]:
tokens = list(text)

vocab = Vocab(tokens)

indices = vocab[tokens[:10]]
print(indices)

recovered = vocab.to_tokens(indices)
print(recovered)

[21, 9, 6, 0, 21, 10, 14, 6, 0, 14]
['t', 'h', 'e', ' ', 't', 'i', 'm', 'e', ' ', 'm']


In [17]:
def build(raw_text, vocab=None):
    """Pipeline hoàn chỉnh: raw text → (corpus, vocab)
    
    Returns:
        corpus: List[int] — toàn bộ text đã encode thành indices
        vocab: Vocab — vocabulary object
    """
    # Bước 1-2: Preprocessing
    text = re.sub('[^A-Za-z]+', ' ', raw_text).lower()
    # Bước 3: Tokenization (character-level)
    tokens = list(text) 
    # Bước 4: Build vocab (nếu chưa có)
    if vocab is None:
        vocab = Vocab(tokens)
    # Encode toàn bộ corpus
    corpus = [vocab[token] for token in tokens]
    return corpus, vocab

corpus, vocab = build(raw_text)
print(f"Corpus length: {len(corpus)}")   # 173428
print(f"Vocab size: {len(vocab)}")       # 28


Corpus length: 173428
Vocab size: 28


In [18]:
class TimeMachineDataset:
    def __init__(self, batch_size, num_steps, num_train=10000, num_val=5000):
        self.batch_size = batch_size
        self.num_steps = num_steps
        self.num_train = num_train
        self.num_val = num_val
        
        # Build corpus và vocab
        corpus, self.vocab = build(raw_text)
        
        # Tạo tất cả subsequences có chiều dài num_steps+1
        # (+1 vì target = input shifted by 1)
        array = torch.tensor([
            corpus[i : i + num_steps + 1] 
            for i in range(len(corpus) - num_steps)
        ])
        
        # Split thành X (input) và Y (target)
        self.X = array[:, :-1]   # Tất cả trừ token cuối
        self.Y = array[:, 1:]    # Tất cả trừ token đầu


In [20]:
data = TimeMachineDataset(batch_size=2, num_steps=10)

X_batch, Y_batch = data.X[:2], data.Y[:2]

print("X_batch shape:", X_batch.shape)  # (2, 10)
print("X_batch:", X_batch)
print("Y_batch shape:", Y_batch.shape)  # (2, 10)
print("Y_batch:", Y_batch)

X_batch shape: torch.Size([2, 10])
X_batch: tensor([[21,  9,  6,  0, 21, 10, 14,  6,  0, 14],
        [ 9,  6,  0, 21, 10, 14,  6,  0, 14,  2]])
Y_batch shape: torch.Size([2, 10])
Y_batch: tensor([[ 9,  6,  0, 21, 10, 14,  6,  0, 14,  2],
        [ 6,  0, 21, 10, 14,  6,  0, 14,  2,  4]])


In [22]:
def get_dataloader(self, train):
    """Tạo dataloader cho X, Y."""
    if train:
        idx = slice(0, self.num_train)
    else:
        idx = slice(self.num_train, self.num_train + self.num_val)

    X_subset = self.X[idx]
    Y_subset = self.Y[idx]

    dataset = torch.utils.data.TensorDataset(X_subset, Y_subset)
    return torch.utils.data.DataLoader(
        dataset, batch_size=self.batch_size, shuffle=train
    )